In [ ]:
#| default_exp ledger

In [ ]:
#| export
from pathlib import Path

In [ ]:
#| export
import numpy as np
from fastcore.all import patch, store_attr
from scipy.optimize import brentq
import pandas as pd

In [ ]:
#| export
from fastcore.all import patch

In [ ]:
from portfolio.util import data_path

## Ledger

A `Ledger` holds the raw truth of a live portfolio: signed transactions and adjusted prices in a common currency. From these it derives all money-weighted analytics — units held, cost basis, market value, P&L, and IRR.

It is deliberately kept separate from `Portfolio` (which handles time-weighted stats). The only bridge between them is `twrr()`, which returns clean asset returns and lagged weights for `Portfolio` to consume.

Sample data for examples and testing. Two assets (gold and equities ETF) with a few transactions including a sell.

In [ ]:
def sample_ledger_data():
    "Generate synthetic transactions and prices for testing"
    idx = pd.date_range('2022-01-01', '2024-06-30', freq='B')
    np.random.seed(42)
    p_gold = 100 * (1 + np.random.normal(0.0003, 0.008, len(idx))).cumprod()
    p_eq   = 100 * (1 + np.random.normal(0.0004, 0.012, len(idx))).cumprod()
    prices = pd.DataFrame(dict(GOLD=p_gold, EQ=p_eq), index=idx)
    prices.index.name = 'date'

    txns = pd.DataFrame([
        dict(date='2022-01-03', symbol='GOLD', units=100,  price=prices.loc['2022-01-03', 'GOLD']),
        dict(date='2022-01-03', symbol='EQ',   units=200,  price=prices.loc['2022-01-03', 'EQ']),
        dict(date='2022-06-01', symbol='EQ',   units=-50,  price=prices.loc['2022-06-01', 'EQ']),
        dict(date='2022-06-01', symbol='GOLD', units=50,   price=prices.loc['2022-06-01', 'GOLD']),
        dict(date='2023-03-01', symbol='EQ',   units=100,  price=prices.loc['2023-03-01', 'EQ']),
    ]).astype({'date': 'datetime64[ns]'})
    txns['tot_invested'] = txns.units * txns.price
    return txns, prices

In [ ]:
txns_s, prices_s = sample_ledger_data()
txns_s

In [ ]:
prices_s.head()

In [ ]:
#| export
class Ledger:
    "Live portfolio ledger: raw transactions + prices in a common currency"
    def __init__(self, txns, prices):
        store_attr()
        wide = txns.pivot_table(index='date', columns='symbol', values=['units', 'tot_invested'], aggfunc='sum')
        self.units_held  = wide['units'].cumsum().reindex(prices.index).ffill()
        self.money_spent = wide['tot_invested'].cumsum().reindex(prices.index).ffill()
        self.market_value = self.units_held * prices
        self.p_l = (self.market_value - self.money_spent).dropna(how='all')
    def __repr__(self): return f"Ledger({list(self.prices.columns)}, {self.prices.index[0].date()} to {self.prices.index[-1].date()})"

Create a ledger from our sample data and inspect it.

In [ ]:
ledger = Ledger(txns_s, prices_s)
ledger

`symbols` gives the list of assets held.

In [ ]:
#| export
@patch(as_prop=True)
def symbols(self:Ledger): return list(self.prices.columns)

In [ ]:
ledger.symbols

## P&L

`p_l` is already computed in `__init__`. `as_of(date)` lets you query the P&L at any point in time — useful for reporting and attribution.

In [ ]:
#| export
@patch
def as_of(self:Ledger, date=None):
    "Total P&L per asset at `date` (default: latest)"
    return self.p_l.asof(date or self.p_l.index[-1])

In [ ]:
# Today's P&L per asset (positive = profit)
ledger.as_of()

In [ ]:
# P&L at a specific date
ledger.as_of('2023-06-01')

In [ ]:
# Total portfolio P&L over time
ledger.p_l.sum(axis=1).plot(title='Total P&L')

## Allocation

`allocation` shows how capital is split across assets at each point in time. Useful for tracking drift and rebalancing decisions.

In [ ]:
#| export
@patch(as_prop=True)
def allocation(self:Ledger):
    "Daily allocation weights (fraction of total market value per asset)"
    return self.market_value.div(self.market_value.sum(axis=1), axis=0).dropna(how='all').round(3)

In [ ]:
# Current allocation
ledger.allocation.iloc[-1]

In [ ]:
# Allocation drift over time
ledger.allocation.resample('MS').last().plot.area(title='Allocation over time')

## IRR (Money-weighted Return)

The IRR answers: *"what annual return did my actual cash, timed the way I timed it, earn?"* Cash flows are `−tot_invested` at each transaction date, plus the current market value as a final inflow.

In [ ]:
#| export
def _npv(r, flows):
    yrs = (flows.index - flows.index[0]).days / 365
    return (flows.values / (1+r)**yrs.values).sum()

In [ ]:
#| export
@patch
def irr(self:Ledger):
    "Annualised money-weighted return (IRR) for the whole portfolio"
    flows = self.txns.groupby('date')['tot_invested'].sum().mul(-1)
    today = self.prices.index[-1]
    flows[today] = flows.get(today, 0) + float(self.market_value.loc[today].sum())
    flows = flows.sort_index()
    return brentq(_npv, -0.9, 10, args=(flows,))

In [ ]:
# Annualised IRR as a decimal (multiply by 100 for %)
ledger.irr()

## Time-weighted Return Bridge

`twrr()` is the single bridge from the money-weighted `Ledger` to the time-weighted `Portfolio`. It returns clean monthly price returns and lagged allocation weights (start-of-month) — free of contribution-timing noise. Assets not yet held or already sold appear as NaN in the weights, so they are excluded from the Portfolio stats during those months.

In [ ]:
#| export
@patch
def twrr(self:Ledger):
    "Monthly asset returns and lagged weights for use with Portfolio"
    asset_rets = self.prices.resample('MS').last().pct_change()
    weights = self.allocation.resample('MS').last().shift().where(lambda w: w != 0)
    return asset_rets, weights

In [ ]:
asset_rets, weights = ledger.twrr()
weights.tail(4).round(3)

In [ ]:
asset_rets.tail(4)

Weights are NaN before the first purchase and after a full sell, so Portfolio stats only use the months where each asset was actually held.

In [ ]:
from portfolio.portfolio import Portfolio

In [ ]:
# Plug into Portfolio for classical time-weighted stats
p_live = Portfolio('live', asset_rets, weights)
p_live.cum_return().plot(title='Time-weighted cumulative return')

This should match up roughly with the money weighted IRR.

In [ ]:
y = (asset_rets.index[-1] - asset_rets.index[0]).days/365
(1+ledger.irr())**y

## Loading from CSV

In production the transactions come from a brokerage CSV (Swedish format here) and prices from pre-downloaded EOD files. `from_csv` handles the messy bits: dtype fixing, FX conversion of EUR-denominated assets to SEK, and forward-filling of missing price days.

In [ ]:
#| export
def _parse_txns(path):
    "Parse brokerage CSV, fix dtypes and sign-encode units"
    txns = pd.read_csv(path)
    txns['Kurs'] = txns['Kurs'].str.replace(',', '.').astype(float)
    txns['Affärsdag'] = pd.to_datetime(txns.Affärsdag)
    txns['Antal'] = txns.Antal.where(txns.Transaktionstyp=='KÖPT', txns.Antal*-1)
    return txns

In [ ]:
#| export
def _load_prices(eod_dir, assets):
    "Load and forward-fill EOD price CSVs for `assets`"
    dfs = [pd.read_csv(Path(eod_dir)/f'{a}.csv', index_col=0, parse_dates=True)[['adjusted_close']].rename(columns=lambda _: a) for a in assets]
    return pd.concat(dfs, axis=1, sort=True)

In [ ]:
#| export
def _to_sek(txns, prices, fx='EURSEK'):
    "Convert EUR-denominated transactions and prices to SEK"
    def eursek(date): return prices.loc[:date, fx].iloc[-1]
    eur = txns['Valuta'] == 'EUR'
    txns = txns.copy()
    eur_assets = txns.loc[eur, 'ISIN'].unique().tolist()
    txns.loc[eur, 'Kurs'] = txns[eur].apply(lambda r: r.Kurs * eursek(r.Affärsdag), axis=1)
    txns.loc[eur, 'Valuta'] = 'SEK'
    prices = prices.copy()
    prices[eur_assets] = prices[eur_assets].multiply(prices[fx], axis=0)
    return txns, prices

In [ ]:
#| export
def _clean_txns(txns):
    "Rename and add tot_invested column"
    tx = txns[['Affärsdag','ISIN','Antal','Kurs']].copy()
    tx.columns = ['date','symbol','units','price']
    tx['tot_invested'] = tx.units * tx.price
    return tx

In [ ]:
#| export
@patch(cls_method=True)
def from_csv(cls:Ledger, txns_path, eod_dir, drop=None, fx='EURSEK'):
    "Build a Ledger from a brokerage transactions CSV and EOD price directory"
    raw = _parse_txns(txns_path)
    assets = list(raw.ISIN.unique()) + [fx]
    prices = _load_prices(eod_dir, assets)
    raw, prices = _to_sek(raw, prices, fx)
    drop = (drop or []) + [fx]
    prices = prices.drop(columns=drop).ffill()
    txns = _clean_txns(raw[~raw.ISIN.isin(drop)])
    return cls(txns, prices)

### A sample csv file (Nordnet broker)

In [ ]:
TEST_DIR = data_path()

In [ ]:
# TEST_DIR/'sample_transactions_test.csv'.write_text(''',Affärsdag,Transaktionstyp,Värdepapper,ISIN,Antal,Kurs,Valuta,Totalt antal
# ,2022-01-03,KÖPT,iShares Core S&P 500 ETF,IE00B5BMR087,100,"95,00",EUR,100
# ,2022-01-03,KÖPT,Swedbank Robur Sverige,SE0000693293,200,"120,00",SEK,200
# ,2022-06-01,SÅLT,iShares Core S&P 500 ETF,IE00B5BMR087,30,"102,50",EUR,70
# ,2023-01-02,KÖPT,iShares Core S&P 500 ETF,IE00B5BMR087,50,"88,00",EUR,120
# ''')

In [ ]:
idx = pd.date_range('2022-01-01', '2023-06-30', freq='B', name='date')
for isin, price in [('IE00B5BMR087', 100), ('SE0000693293', 120), ('EURSEK', 10)]:
    pd.Series(price, index=idx, name='close').to_csv(TEST_DIR / f'{isin}.csv')

In [ ]:
ledger_t = Ledger.from_csv(TEST_DIR/'sample_transactions_test.csv', TEST_DIR)
ledger_t

P&L For SEK asset is 0 since price has not moved.

In [ ]:
ledger_t.as_of()

EUR asset should be converted to SEK; SEK asset stays as-is

In [ ]:
ledger_t.txns

Units held: IE00B5BMR087 should be 120 (100 - 30 + 50), SE0000693293 should be 200

In [ ]:
ledger_t.units_held.iloc[-1]

## A 100% buy and sell scenario

In [ ]:
def buy_sell_data():
    idx = pd.date_range('2022-01-03', '2023-01-03', freq='B')
    np.random.seed(7)
    px = 100 * (1 + np.random.normal(0.0005, 0.01, len(idx))).cumprod()
    prices = pd.DataFrame(dict(ASSET=px), index=idx)
    prices.index.name = 'date'
    txns = pd.DataFrame([
        dict(date='2022-01-03', symbol='ASSET', units=100,  price=prices.loc['2022-01-03','ASSET']),
        dict(date='2023-01-03', symbol='ASSET', units=-100, price=prices.loc['2023-01-03','ASSET']),
    ]).astype({'date': 'datetime64[ns]'})
    txns['tot_invested'] = txns.units * txns.price
    return txns, prices

txns_bs, prices_bs = buy_sell_data()
txns_bs

Build the ledger and inspect units held — should drop to zero on the sell date.

In [ ]:
ledger_bs = Ledger(txns_bs, prices_bs)
ledger_bs.units_held.tail(3)

`money_spent` after the sell should be negative — net proceeds exceed cost, so net cash flow is positive to us.

In [ ]:
ledger_bs.money_spent.tail(3)

P&L over the holding period. After the sell, `market_value = 0` so `p_l = -money_spent = proceeds - cost`. This is the locked-in realized gain.

In [ ]:
ledger_bs.p_l.sum(axis=1).plot(title='P&L: buy then full sell')

Verify the final P&L matches the simple calculation: sell proceeds − buy cost.

In [ ]:
buy_cost     = txns_bs.loc[0, 'tot_invested']
sell_proceeds = -txns_bs.loc[1, 'tot_invested']
realized_pnl = sell_proceeds - buy_cost
ledger_pnl   = (ledger_bs.p_l.sum(axis=1).iloc[-1])
realized_pnl, ledger_pnl

IRR after a full sell — reflects the annual return on the completed trade only. This is the price difference between buy and sell date.

In [ ]:
ledger_bs.irr(), txns_bs.price.pct_change().dropna().item()